In [ ]:
!pip tensorflow transformers

ERROR: unknown command "tensorflow"


In [ ]:
!pip install pandas scikit-learn

In [ ]:
import pandas as pd
import regex as re
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, TFBertForSequenceClassification
import tensorflow as tf


In [ ]:
data = pd.read_csv('/content/IMDB Dataset.csv', engine='python', on_bad_lines='warn')

def clean_text(text):
    text = re.sub(r'<.*?>', '', text)  # remove HTML tags like <br />
    text = text.strip()                # remove leading/trailing spaces
    return text

data['review'] = data['review'].apply(clean_text)


print(data.head())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. The filming tec...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [ ]:
print(data.shape)

(26078, 2)


In [ ]:
print(data['review'][0])

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.I would say the main appeal of the show is due to the fact that it goes where other shows wou

avarage-len-of-reviews(in-words)

In [ ]:
avg_words = data['review'].apply(lambda x: len(str(x).split())).mean()
print(avg_words)


227.45781885113888


In [ ]:
data['sentiment'] = data['sentiment'].map({'positive': 1, 'negative': 0})

X = data['review'].values
y = data['sentiment'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
print(X_train)

["This Book-based movie is truly awful, and a big disappointment. We've been waiting for this move over a month. Many film reviewer were hopeful for it. Also in newspapers and TV, it made big sense. When 29th April comes, many people regretfully noticed that movie is really awful. Why? First of all story was so monotone. It has been many indefinite scenes, sometimes it's hard to realize what's going on. The actresses, out of Hulya Avsar, weren't harmonized with their roles, especially Vildan Atasever. She acts better in comedy films, In this movie, a kind of drama, she couldn't disposed of her previous role. And finally Movie is too short, just 66 minutes."
 "The wind and the lion is a marvelous sweeping motion picture. It is a monument to what filmmaking once was but is no more.Connery, despite the thick scottish brogue, plays the Raisulu very well. He inspires the viewer in a way many lead characters cannot. Candice Bergen, in one of her early roles, is marvelous as the kidnapped soc

In [ ]:
print(y_train)

[0 1 0 ... 1 1 1]


In [ ]:
# Load BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = TFBertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2, use_safetensors=False)
# Tokenize the training data
train_encodings = tokenizer(list(X_train), truncation=True, padding=True, max_length=256, return_tensors='tf')
test_encodings = tokenizer(list(X_test), truncation=True, padding=True, max_length=256, return_tensors='tf')
# Create TensorFlow datasets
train_dataset = tf.data.Dataset.from_tensor_slices((dict(train_encodings), y_train)).shuffle(1000).batch(8)
test_dataset = tf.data.Dataset.from_tensor_slices((dict(test_encodings), y_test)).batch(8)
# Compile and train the model
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])
# Train the model
model.fit(train_dataset, epochs=3, validation_data=test_dataset)

# After training:
model.save_pretrained("my_finetuned_model")
tokenizer.save_pretrained("my_finetuned_model")

tf_model.h5:   0%|          | 0.00/536M [00:00<?, ?B/s]

All model checkpoint layers were used when initializing TFBertForSequenceClassification.

Some layers of TFBertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


Epoch 1/3
2608/2608 [==============================] - 1353s 498ms/step - loss: 0.3006 - accuracy: 0.8716 - val_loss: 0.2671 - val_accuracy: 0.9036
Epoch 2/3
2608/2608 [==============================] - 1290s 495ms/step - loss: 0.1770 - accuracy: 0.9333 - val_loss: 0.3044 - val_accuracy: 0.8815
Epoch 3/3
2608/2608 [==============================] - 1290s 494ms/step - loss: 0.1087 - accuracy: 0.9610 - val_loss: 0.2766 - val_accuracy: 0.9114


('my_finetuned_model/tokenizer_config.json',
 'my_finetuned_model/special_tokens_map.json',
 'my_finetuned_model/vocab.txt',
 'my_finetuned_model/added_tokens.json')

In [ ]:
!zip -r my_finetuned_model.zip my_finetuned_model

  adding: my_finetuned_model/ (stored 0%)
  adding: my_finetuned_model/special_tokens_map.json (deflated 42%)
  adding: my_finetuned_model/config.json (deflated 48%)
  adding: my_finetuned_model/vocab.txt (deflated 53%)
  adding: my_finetuned_model/tf_model.h5 (deflated 8%)
  adding: my_finetuned_model/tokenizer_config.json (deflated 75%)


In [ ]:
#  Example review text
text = "This movie was absolutely amazing! The actors did a great job."

#  Tokenize the input
inputs = tokenizer(text, return_tensors='tf', truncation=True, padding=True, max_length=256)

#  Get model predictions
outputs = model(**inputs)

#  Convert logits → probabilities → predicted label
probs = tf.nn.softmax(outputs.logits, axis=1)
pred_class = tf.argmax(probs, axis=1).numpy()[0]
if pred_class == 1:
    pred_class = "positive"
else:
    pred_class = "negative"

#  Print result
print("Probabilities:", probs.numpy())
print("Predicted class:", pred_class)

Probabilities: [[0.00375116 0.9962489 ]]
Predicted class: positive
